# FaceProof 03 — train probe + evaluate (C4, in-distribution)

**Day 2 gate (part 2):** fit the **C4** detector — a logistic-regression probe on frozen CLIP
features — and confirm it separates real vs StyleGAN **in-distribution** (AUROC > ~0.9).

The 'detector' is just logistic regression on cached features: no backbone training, fits in
seconds. This same probe gives calibrated probabilities we'll reuse for calibration (Day 6).

> **Day 3 (next):** evaluate this probe on the *unseen* generator (SD, the `test` split) and add
> the **C1** ResNet probe → the cross-generator generalization gap table (RQ1).

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "day1-preprocess-notebook"   # or "main" once merged
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q scikit-learn pyyaml
import sys, os
sys.path.append(os.getcwd())

## 2. Load manifest + cached CLIP features

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')

ROOT          = Path('/content/drive/MyDrive/faceproof')
FEATURES_ROOT = ROOT / 'models' / 'features'
MANIFEST      = ROOT / 'data' / 'manifest.csv'

df = pd.read_csv(MANIFEST)
X_clip = np.load(FEATURES_ROOT / 'clip_all.npy')   # made in notebook 02
assert len(df) == len(X_clip), 'manifest and features must align (re-run notebook 02)'
print('rows:', len(df), '| CLIP dim:', X_clip.shape[1])
print(df['split'].value_counts())

## 3. Train the C4 probe (in-distribution)

Fit on the **train** split only (real + StyleGAN). Evaluate on **test_indist** = held-out
StyleGAN the probe never saw. Same generator family = *in-distribution*.

In [ ]:
from src.probe import train_probe, predict_proba

# Boolean masks over manifest rows (features are in the same order).
m_train   = (df['split'] == 'train').values
m_indist  = (df['split'] == 'test_indist').values     # held-out StyleGAN = in-distribution test
y = df['label'].values

Xtr, ytr = X_clip[m_train],  y[m_train]
Xin, yin = X_clip[m_indist], y[m_indist]
print(f'train: {len(ytr)}  (real={ (ytr==0).sum() }, fake={ (ytr==1).sum() })')
print(f'in-dist test: {len(yin)}')

clf = train_probe(Xtr, ytr)            # standardize + logistic regression, fits in seconds
p_in = predict_proba(clf, Xin)         # P(synthetic) for in-distribution test

## 4. ✅ Day 2 gate — in-distribution AUROC

**AUROC** = probability the probe ranks a random fake above a random real (1.0 perfect, 0.5
chance). **EER** = error rate where false-accepts equal false-rejects (lower is better).

In [ ]:
from src.metrics import auroc, eer

auc = auroc(yin, p_in)
err = eer(yin, p_in)
print(f'C4 (CLIP) in-distribution  AUROC = {auc:.4f}  |  EER = {err:.4f}')
print()
if auc > 0.90:
    print(f'✅ Day 2 gate PASSED: in-distribution AUROC {auc:.4f} > 0.90.')
    print('   The frozen-CLIP probe clearly separates real vs StyleGAN in-distribution.')
else:
    print(f'⚠️  in-distribution AUROC {auc:.4f} < 0.90 — investigate before moving on')
    print('   (check feature alignment, class balance, or preprocessing).')

---
**Log it.** Add a row to `experiments.md`: `C4 | StyleGAN | StyleGAN (in-dist) | none | seed 13 | AUROC | <value>`,
plus the Day 2 daily lines (leakage acc + in-dist AUROC).

**Day 3 next:** evaluate `clf` on the `test` split (unseen SD) for the cross-generator number,
then repeat C1 with `resnet_all.npy` to build the generalization-gap table.